In [1]:
1

1

In [2]:
from fetcing_data_settings import LoadData
import json
import pandas as pd

LOADING DATA 

In [3]:
with open('all_review.json','r',encoding="utf-8") as f:
    data = json.load(f)

flat_data = []
for group in data:
    if group:
        for review in group:
            if review:
                flat_data.append(review)


df = pd.DataFrame(flat_data)
df

,id,customer_name,score,message,created_at
0,136df76d-36d7-47c0-9dd9-38c1de3990a6,Sebine,1,,2026-04-22T07:50:35.493198Z
1,d786f52b-8827-447f-9cbb-efc21c43dd53,,5,super👍👍,2026-04-17T04:25:39.402822Z
2,8df6833a-b6bc-430a-84d5-5af18b646971,Sevil,5,çox razıyam,2026-04-08T14:05:29.601609Z
3,ed8ec82c-eaef-4898-b8d5-886b63b56654,lamiye haciyeva,5,Əlaa,2026-04-07T19:01:03.092897Z
4,676ba2e8-eead-4acc-80be-ba185a60b68f,,5,rəngi bir tık tünd götürmədiyimə peşmanam 🥲 Be...,2026-04-05T19:18:11.144172Z
...,...,...,...,...,...
4268,36e16224-9fe4-4afe-8b17-fc08affb0b56,Jalə,3,,2026-04-23T15:30:26.765569Z
4269,1fc6f5fe-72fe-4a07-a49f-0edd684d6882,,5,10 günə 5 kq arıqladım. Superdi👍,2026-04-19T15:12:58.615455Z
4270,4747c1ee-2f41-4f80-80c5-f9b77da7617f,Fatimə Əlizadə,5,1 qutu ilə 6 kq arıqladımm😍,2026-03-06T18:23:20.26626Z
4271,b4c28c4e-ae1d-4169-8a0a-9f2799afa141,,5,Super,2026-03-04T14:37:42.376494Z


CLEANING DATA

In [4]:
df = df[df['message'] != ''] 
df = df[['score','message']]

CHECKING SCORE ACCURACY

In [79]:
# if review doesnt match score we can fix it in excel , and then update true score
positive_words = ["yaxsi","yaxşı","ela","super","qeseng","pis deyil"]
negative_words = ["pis","berbad","rezalet"]

def flag_row(row):
    text = str(row["message"]).lower()
    score = row["score"]

    if len(text.strip()) <= 2:
        return "too_short"

    if score == 3 and any(w in text for w in positive_words):
        return "neutral_but_positive"

    if score >= 4 and any(w in text for w in negative_words):
        return "high_score_negative_text"

    if score <= 2 and any(w in text for w in positive_words):
        return "low_score_positive_text"

    return None

df["flag"] = df.apply(flag_row, axis=1)

suspects = df[df["flag"].notna()].copy()



suspects = suspects[["message", "score", "flag"]].reset_index()
suspects.columns = ["row_id", "message", "score", "flag"]
suspects["new_score"] = ""
suspects["action"] = ""

display(suspects.head(10))

suspects.to_csv("suspects.csv", index=False, encoding="utf-8-sig")

,row_id,message,score,flag,new_score,action
0,60,Almağa dəyər yaxşı çıxdı😊,2,low_score_positive_text,,
1,69,Yaxşıdı,3,neutral_but_positive,,
2,78,nezikdi ama istifade si asantdi pis deyil,5,high_score_negative_text,,
3,82,ince nazikdi biraz. amma qiymetine gore pis deyil,3,neutral_but_positive,,
4,127,Heç də yaxşı nəmləndirmir. Dəyməz bu qiymətə,1,low_score_positive_text,,
5,434,elede yaxsi deil beyenmedim,1,low_score_positive_text,,
6,474,"290 bar yoxdur burda, maksimum 130-140 olar. K...",3,neutral_but_positive,,
7,595,Pis deyil İDARƏ edər,5,high_score_negative_text,,
8,624,pis deyil,3,neutral_but_positive,,
9,654,Keyfiyyəti pis deyil. Lakin tüklərinin hündürl...,3,neutral_but_positive,,


In [69]:
df['score_original'] = df['score']

reviewed = pd.read_csv('suspects.csv')

reviewed.columns = reviewed.columns.str.strip()
reviewed["action"] = reviewed["action"].astype(str).str.strip().str.lower()
reviewed["new_score"] = pd.to_numeric(reviewed["new_score"], errors="coerce")


updates = reviewed[(reviewed["action"] == 'update') & (reviewed["new_score"].notna()) ]


for _,row in updates.iterrows():
    idx = int(row['row_id'])
    df.loc[idx, "score"] = int(row["new_score"])


deletes = reviewed[(reviewed["action"] == 'delete')]

for _,row in updates.iterrows():
    idx = int(row['row_id'])
    df.drop(index=idx, inplace=True)


df.reset_index(drop=True)

df["score"].value_counts().sort_index()




score
1     175
2      38
3      60
4     132
5    2088
Name: count, dtype: int64

DATA IS TOO MUCH IMBALANCE SO WE MAKE IT BINARY


In [86]:
df_bin = df[df['score'] != 3]
df_bin['label'] = df_bin['score'].apply(lambda x : 0 if x <= 2 else 1)

final_df = df_bin[["message", "label"]].rename(columns={"message": "text"})
final_df = final_df.reset_index(drop=True)




CHECKING FINAL DATA FRAME

In [87]:
final_df["label"].value_counts()


label
1    2220
0     213
Name: count, dtype: int64

In [88]:
final_df["text"].isna().sum()


np.int64(0)

In [89]:
final_df["text"].duplicated().sum()

np.int64(375)

In [90]:
dup_check = final_df.groupby("text")["label"].nunique()

conflict_texts = dup_check[dup_check > 1]

print("Conflict duplicate count:", len(conflict_texts))

Conflict duplicate count: 4


FIXING CONFLICT REVIEWS

In [91]:
conflict_rows = final_df[
    final_df["text"].isin(conflict_texts.index)
].sort_values("text")

conflict_rows.shape

(54, 2)

In [92]:
positive_texts = conflict_rows['text'].str.lower().str.strip().unique().tolist()

final_df.loc[
    final_df["text"].str.lower().str.strip().isin(positive_texts),
    "label"
] = 1

In [93]:
dup_check = final_df.groupby("text")["label"].nunique()
print((dup_check > 1).sum())

0


In [94]:
final_df.to_csv("sentiment_binary_full.csv", index=False)